# Smoke-pair split audit (read-only)
Finds which of the smoke-test pairs fall in the project's frozen **test** split (hash-based, from `src/data.py`), and writes a cleaned CSV without them.
Nothing in the repo is modified. No GPU needed (CPU runtime is fine).

In [ ]:
!pip -q install datasets pandas
!git clone https://github.com/tanvishetty1403/pessimistic-rlhf-beta-audit.git
%cd pessimistic-rlhf-beta-audit

## 1. Upload `smoke_gold_scores.csv`

In [ ]:
import os
from google.colab import files
if not os.path.exists("smoke_gold_scores.csv"):
    files.upload()  # pick smoke_gold_scores.csv
print(os.path.exists("smoke_gold_scores.csv"))

## 2. Assign each smoke pair to the project's split

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import pandas as pd
from datasets import load_dataset
from src.data import extract_prompt, assign_split

SEED = 0  # matches configs/base.yaml

d = pd.read_csv("smoke_gold_scores.csv")
ds = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base", split="train")
rows = d["id"].str.split("-").str[1].astype(int)
assert rows.max() < len(ds), "ID out of range for helpful-base train"

# split depends only on the prompt hash, so the choice of subset does not matter
d["project_split"] = [
    assign_split(extract_prompt(ds[int(r)]["chosen"]), SEED) for r in rows
]
print("pairs per project split:")
print(d["project_split"].value_counts().to_string())

## 3. Results (diagnostic only; do not report the test-split row)

In [ ]:
print(d.groupby("project_split")["gold_agrees"].agg(["mean", "count"]).round(4).to_string())

safe = d[d["project_split"] != "test"].drop(columns="project_split")
safe.to_csv("smoke_gold_scores_no_test.csv", index=False)
print("\nwrote smoke_gold_scores_no_test.csv:", len(safe), "pairs")
print("agreement WITHOUT test-split pairs: %.4f" % safe["gold_agrees"].mean())

## 4. Download the cleaned CSV
Share **only** this file. Do not commit the original CSV.

In [ ]:
files.download("smoke_gold_scores_no_test.csv")